# Raw Data Exploration

In [0]:
file_location = "s3://my-datasets-for-demo/customer-churn-datasets/bronze/online_retail_II_2009-2010.csv"
df = spark.read.csv(file_location, header=True, inferSchema=True)
df.show(10)

+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|     InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|01/12/2009 07:45| 6.95|      13085|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|01/12/2009 07:45| 6.75|      13085|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|01/12/2009 07:45| 6.75|      13085|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|01/12/2009 07:45|  2.1|      13085|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|01/12/2009 07:45| 1.25|      13085|United Kingdom|
| 489434|    22064|PINK DOUGHNUT TRI...|      24|01/12/2009 07:45| 1.65|      13085|United Kingdom|
| 489434|    21871| SAVE THE PLANET MUG|      24|01/12/2009 07:45| 1.25|      13085|United Kingdom|


In [0]:
file_location_2 = "s3://my-datasets-for-demo/customer-churn-datasets/bronze/online_retail_II_2010-2011.csv"
df_2 = spark.read.csv(file_location_2, header=True, inferSchema=True)
df_2.show(10)

+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|     InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
| 536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 08:26| 2.55|      17850|United Kingdom|
| 536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 08:26| 3.39|      17850|United Kingdom|
| 536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 08:26| 2.75|      17850|United Kingdom|
| 536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 08:26| 3.39|      17850|United Kingdom|
| 536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 08:26| 3.39|      17850|United Kingdom|
| 536365|    22752|SET 7 BABUSHKA NE...|       2|01/12/2010 08:26| 7.65|      17850|United Kingdom|
| 536365|    21730|GLASS STAR FROSTE...|       6|01/12/2010 08:26| 4.25|      17850|United Kingdom|


## Data Types

In [0]:
# Check data types
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
# Check data types
df_2.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)



## Data Dimensions

In [0]:
# Check shape
print(df.count(), len(df.columns))
print(df_2.count(), len(df_2.columns))

525461 8
541910 8


In [0]:
# Combine 2 dataframes into one df
df = df.union(df_2)

In [0]:
from pyspark.sql.functions import to_timestamp

# Fix data type
df = df \
    .withColumn("InvoiceDate", to_timestamp("InvoiceDate", "dd/MM/yyyy HH:mm"))
print(df.count(), len(df.columns))
df.printSchema()

1067371 8
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)



## Missing Values

In [0]:
from pyspark.sql.functions import col, count, isnan, when

# Check missing values
# Count NULL values in each column
df.select(
    [
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ]
).show()

# Count NAN values in each column
df.select(
    [
        count(when(isnan(col(c)), c)).alias(c) 
        for c, dtype in df.dtypes
        if dtype == 'double'
    ]
).show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|       4382|       0|          0|    0|     243007|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+

+-----+
|Price|
+-----+
|    0|
+-----+



## Duplicate

In [0]:
print(df.count() - df.dropDuplicates().count())

34335


## Count Distinct Values

In [0]:
from pyspark.sql.functions import col, countDistinct

df.select(
    [
        countDistinct(col(c), ).alias(c) 
        for c in df.columns
    ]
).show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|  53628|     5305|       5698|    1057|      47635| 2807|       5942|     43|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



## Check Invalid Values

In [0]:
from pyspark.sql.functions import col, count, when

df.select(
    count(when(col("Quantity") <= 0, 1)).alias("Invalid_Quantities"),
    count(when(col("Price") <= 0, 1)).alias("Invalid_Prices"),
    count(when(col("Customer ID") <= 0, 1)).alias("Negative_Customer_IDs")
).show()

df.select(
    count(when(col("Quantity")==0, 1)).alias("Zero_Quantity"),
    count(when(col("Quantity")<0, 1)).alias("Negative_Quantity"),
    count(when(col("Price")<0, 1)).alias("Negative_Price"),
    count(when(col("Price")==0, 1)).alias("Zero_Price")
).show()


+------------------+--------------+---------------------+
|Invalid_Quantities|Invalid_Prices|Negative_Customer_IDs|
+------------------+--------------+---------------------+
|             22950|          6207|                    0|
+------------------+--------------+---------------------+

+-------------+-----------------+--------------+----------+
|Zero_Quantity|Negative_Quantity|Negative_Price|Zero_Price|
+-------------+-----------------+--------------+----------+
|            0|            22950|             5|      6202|
+-------------+-----------------+--------------+----------+



In [0]:
from pyspark.sql.functions import col

# negative_rows_df = df.filter(
#     (col("Quantity") <= 0) | 
#     (col("Price") <= 0)
# )

negative_rows_df = df.filter(
    (col("Price") <= 0)
)
negative_rows_df.show()


+-------+---------+-------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|        Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+-------------------+--------+-------------------+-----+-----------+--------------+
| 489464|    21733|       85123a mixed|     -96|2009-12-01 10:52:00|  0.0|       NULL|United Kingdom|
| 489463|    71477|              short|    -240|2009-12-01 10:52:00|  0.0|       NULL|United Kingdom|
| 489467|   85123A|        21733 mixed|    -192|2009-12-01 10:53:00|  0.0|       NULL|United Kingdom|
| 489521|    21646|               NULL|     -50|2009-12-01 11:44:00|  0.0|       NULL|United Kingdom|
| 489655|    20683|               NULL|     -44|2009-12-01 17:26:00|  0.0|       NULL|United Kingdom|
| 489659|    21350|               NULL|     230|2009-12-01 17:39:00|  0.0|       NULL|United Kingdom|
| 489660|    35956|               lost|   -1043|2009-12-01 17:43:00|  0.0|       N

In [0]:
from pyspark.sql.functions import col, count, sum, trim, upper, when

# Check cancelled transactions
df.filter(col("Invoice").startswith("C")).show(10)

clean_invoice = upper(trim(col("Invoice")))
df.select(
    count(
        when(clean_invoice.startswith("C") & (col("Quantity") < 0), 1)
    ).alias("c_and_negative_qty_count"),
    count(
        when(clean_invoice.startswith("C") & (col("Quantity") > 0), 1)
    ).alias("c_with_positive_qty_count"),
    count(
        when(clean_invoice.startswith("C") & (col("Quantity") < 0) & (col("Price") == 0), 1)
    ).alias("c_with_negative_qty_count_and_zero_price")
).show()

df.select(
    count(when(clean_invoice.rlike("[^0-9]"), 1)).alias("non_numeric_invoice_count"),
    count(when(col("Quantity") < 0, 1)).alias("negative_quantity_count"),
    count(when(clean_invoice.rlike("[^0-9]") & (col("Quantity") < 0), 1)).alias("both_count")
).show()

df.filter(
    ~clean_invoice.startswith("C") & clean_invoice.rlike("[^0-9]")
).show()

df.filter(
    clean_invoice.startswith("C") & (col("Quantity") > 0)
).show()

+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|     InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+----------------+-----+-----------+--------------+
|C489449|    22087|PAPER BUNTING WHI...|     -12|01/12/2009 10:33| 2.95|      16321|     Australia|
|C489449|   85206A|CREAM FELT EASTER...|      -6|01/12/2009 10:33| 1.65|      16321|     Australia|
|C489449|    21895|POTTING SHED SOW ...|      -4|01/12/2009 10:33| 4.25|      16321|     Australia|
|C489449|    21896|  POTTING SHED TWINE|      -6|01/12/2009 10:33|  2.1|      16321|     Australia|
|C489449|    22083|PAPER CHAIN KIT R...|     -12|01/12/2009 10:33| 2.95|      16321|     Australia|
|C489449|    21871| SAVE THE PLANET MUG|     -12|01/12/2009 10:33| 1.25|      16321|     Australia|
|C489449|    84946|ANTIQUE SILVER TE...|     -12|01/12/2009 10:33| 1.25|      16321|     Australia|


In [0]:
# Check InvoiceDate range
from pyspark.sql.functions import min, max

df.select(
    min("InvoiceDate").alias("Earliest_Invoice_Date"),
    max("InvoiceDate").alias("Latest_Invoice_Date")
).show()

+---------------------+-------------------+
|Earliest_Invoice_Date|Latest_Invoice_Date|
+---------------------+-------------------+
|  2009-12-01 07:45:00|2011-12-09 12:50:00|
+---------------------+-------------------+



In [0]:
# Show all distinct values in the Country column
df.select("Country").distinct().show(43)

+--------------------+
|             Country|
+--------------------+
|         Switzerland|
|               Italy|
|              Norway|
|             Austria|
|              Poland|
|         Netherlands|
|         Unspecified|
|              Greece|
|           Australia|
|              France|
|United Arab Emirates|
|              Sweden|
|             Denmark|
|                 USA|
|             Finland|
|     Channel Islands|
|                EIRE|
|             Nigeria|
|      United Kingdom|
|               Japan|
|            Portugal|
|              Cyprus|
|             Belgium|
|             Germany|
|               Spain|
|             Bermuda|
|             Bahrain|
|               Malta|
|                 RSA|
|           Hong Kong|
|           Singapore|
|              Israel|
|            Thailand|
|         West Indies|
|           Lithuania|
|              Brazil|
|               Korea|
|             Lebanon|
|              Canada|
|             Iceland|
|        Sa